# Dijkstra's Single-Source Shortest-Path Algorithm: An Interactive Walkthrough

This notebook turns the lecture on **Dijkstra's algorithm** into a hands-on, executable demo. You will build the example graph, implement the core primitives, watch the greedy main loop run step-by-step, benchmark two priority-queue implementations, and see exactly why the algorithm breaks on negative edges.

## Learning objectives

1. Explain the single-source shortest-path problem and the **greedy** strategy Dijkstra uses on graphs with **non-negative** edge weights.
2. Implement **Initialize-Single-Source** and **edge relaxation**, and describe how relaxation drives distance estimates toward the true shortest-path distances.
3. **Trace** the main loop vertex-by-vertex, distinguishing finalized vertices `S`, the vertex `u` being processed, and the queue `Q = V - S`.
4. **Compare** running times under linear-array, binary-heap, and Fibonacci-heap priority queues — and confirm the array-vs-heap trade-off empirically.
5. State the **loop invariant** and explain, via a negative-edge counterexample, why correctness depends on non-negative weights.

## Roadmap

| Movement | Cells | Topic |
|---|---|---|
| Setup | C01–C02 | Imports and environment |
| Problem & relaxation | C03–C07 | SSSP problem, `relax`, interactive demo |
| Main loop & visualization | C08–C10 | Traced Dijkstra + scrubbable animations |
| Complexity | C11–C12 | Priority-queue analysis + timing |
| Correctness | C13–C14 | Loop invariant + negative-edge counterexample |
| Recap | C15 | Summary and next steps |

Run the cells **top-to-bottom** — each builds on the previous one.

In [ ]:
# C02 — Environment setup: install if needed, then import all dependencies.
# Every import lives here so downstream cells run top-to-bottom without re-importing.
import sys, subprocess

def _pip_install(pkg):
    print(f"Installing {pkg} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import heapq, math, time, random, itertools

try:
    import networkx as nx
except ImportError:
    _pip_install("networkx")
    import networkx as nx

try:
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError:
    _pip_install("matplotlib")
    import matplotlib
    import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, Dropdown, Checkbox, Play, jslink
except ImportError:
    _pip_install("ipywidgets")
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, Dropdown, Checkbox, Play, jslink

# Enable inline plotting only inside an IPython/Colab runtime.
try:
    _ip = get_ipython()
except NameError:
    _ip = None
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")

# Reproducible random graphs for the timing experiment (C12).
random.seed(42)

# Sanity checks: confirm every module imported and ipywidgets is functional.
for _name, _mod in [("networkx", nx), ("matplotlib", matplotlib), ("ipywidgets", widgets)]:
    assert _mod is not None, f"Failed to import {_name}"
_ = IntSlider()
print(f"Setup OK — networkx {nx.__version__}, matplotlib {matplotlib.__version__}")

## The single-source shortest-path (SSSP) problem

We are given a **weighted, directed graph** $G = (V, E)$ with a weight $w(u, v) \ge 0$ on every edge, and a **source** vertex $s$. The goal: find a shortest (minimum-weight) path from $s$ to **every** other vertex.

### Notation

| Symbol | Meaning |
|---|---|
| $w(u, v)$ | weight of edge $u \to v$ (assumed $\ge 0$) |
| $\delta(s, v)$ | **true** shortest-path distance from $s$ to $v$ |
| $v.d$ | current **estimate** of the distance (always $\ge \delta(s,v)$) |
| $v.\pi$ | **predecessor** of $v$ on the current shortest path (for reconstruction) |
| $S$ | set of **finalized** vertices (estimate proven correct) |
| $Q = V - S$ | vertices still waiting in the priority queue |

### The greedy idea

Dijkstra repeatedly **picks the unfinalized vertex with the smallest estimate** $v.d$, declares it finalized (moves it from $Q$ into $S$), and **relaxes** its outgoing edges. Because all weights are non-negative, the nearest unfinalized vertex can never be improved later — so the greedy choice is safe.

> **Connection to Prim's algorithm.** Structurally Dijkstra is nearly identical to Prim's MST algorithm: both grow a set $S$ by repeatedly extracting the closest fringe vertex from a priority queue. The difference is only the key being minimized — path distance from $s$ here, versus single edge weight in Prim.

In [ ]:
# C04 — The lecture example graph + the shared draw_graph visualization helper.
COLOR_S = "black"        # finalized vertices (in S)
COLOR_U = "gray"         # the vertex currently being processed (u)
COLOR_Q = "white"        # vertices still in the queue (Q = V - S)
COLOR_PRED_EDGE = "crimson"

# Slide-3 example graph. Final distances: {s:0, y:5, t:8, z:7, x:9}.
EXAMPLE_EDGES = [
    ("s", "t", 10), ("s", "y", 5),
    ("t", "x", 1),  ("t", "y", 2),
    ("y", "t", 3),  ("y", "x", 9), ("y", "z", 2),
    ("x", "z", 4),
    ("z", "x", 6),  ("z", "s", 7),
]

POS = {
    "s": (0.0, 1.0),
    "t": (1.0, 2.0),
    "y": (1.0, 0.0),
    "x": (2.0, 2.0),
    "z": (2.0, 0.0),
}

def build_graph(edge_list):
    g = nx.DiGraph()
    for u, v, w in edge_list:
        g.add_edge(u, v, weight=w)
    return g

def _fmt_dist(value):
    '''Render math.inf (and None) as the infinity symbol.'''
    if value is None or (isinstance(value, float) and math.isinf(value)):
        return "∞"
    return str(value)

def draw_graph(graph, pos, dist=None, pi=None, S=None, u=None, Q=None, title=None, ax=None):
    '''Draw `graph` colouring S=black, u=gray, Q=white, highlighting predecessor edges.

    The single visualization primitive reused by C09, C10 and C14. Returns the Axes.
    '''
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    S = set(S) if S is not None else set()
    if Q is None:
        Q = set(graph.nodes()) - S - ({u} if u is not None else set())
    # Fall back to a deterministic layout if any node lacks a fixed position.
    if any(n not in pos for n in graph.nodes()):
        pos = nx.spring_layout(graph, seed=42)

    node_colors = []
    for n in graph.nodes():
        if n == u:
            node_colors.append(COLOR_U)
        elif n in S:
            node_colors.append(COLOR_S)
        else:
            node_colors.append(COLOR_Q)

    # Predecessor edges form the current shortest-path tree.
    pred_edges = set()
    if pi:
        for v, p in pi.items():
            if p is not None and graph.has_edge(p, v):
                pred_edges.add((p, v))
    other_edges = [e for e in graph.edges() if e not in pred_edges]

    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, edgecolors="black",
                           node_size=1200, ax=ax)
    nx.draw_networkx_edges(graph, pos, edgelist=other_edges, edge_color="lightgray",
                           arrows=True, arrowstyle="->", arrowsize=15,
                           connectionstyle="arc3,rad=0.08", node_size=1200, ax=ax)
    if pred_edges:
        nx.draw_networkx_edges(graph, pos, edgelist=list(pred_edges), edge_color=COLOR_PRED_EDGE,
                               width=2.5, arrows=True, arrowstyle="->", arrowsize=18,
                               connectionstyle="arc3,rad=0.08", node_size=1200, ax=ax)

    # Node labels: name plus distance estimate (inf-safe). White text on dark nodes.
    for n in graph.nodes():
        x, y = pos[n]
        txt_color = "white" if (n in S or n == u) else "black"
        label = n if dist is None else f"{n}\n{_fmt_dist(dist.get(n))}"
        ax.text(x, y, label, ha="center", va="center", color=txt_color,
                fontsize=10, fontweight="bold")

    edge_labels = {(a, b): str(d.get("weight", "?")) for a, b, d in graph.edges(data=True)}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8,
                                 label_pos=0.4, ax=ax)

    ax.set_title(title or "Graph")
    ax.set_axis_off()
    return ax

example_graph = build_graph(EXAMPLE_EDGES)

# Tests
assert set(example_graph.nodes()) == {"s", "t", "x", "y", "z"}
assert example_graph.number_of_edges() == len(EXAMPLE_EDGES)
assert all(n in POS for n in example_graph.nodes())

draw_graph(example_graph, POS, title="Lecture example graph (initial state)")
plt.show()
print(f"Example graph: {example_graph.number_of_nodes()} nodes, "
      f"{example_graph.number_of_edges()} edges")

## Initialize-Single-Source and Relax

Two primitives power the whole algorithm.

**Initialize-Single-Source($G$, $s$)** sets every estimate to $\infty$ except the source:

```
INITIALIZE-SINGLE-SOURCE(G, s)
  for each vertex v in G.V:
      v.d   = infinity
      v.pi  = NIL
  s.d = 0
```

**Relax($u$, $v$, $w$)** is the **atomic update** — it asks "is going to $v$ *through* $u$ cheaper than the best route to $v$ we know so far?":

```
RELAX(u, v, w)
  if v.d > u.d + w(u, v):
      v.d  = u.d + w(u, v)     # found a shorter route to v
      v.pi = u                  # record predecessor for path reconstruction
```

Relaxation is the only operation that ever changes a distance estimate. It is **monotone**: $v.d$ can only decrease, and it never drops below the true distance $\delta(s, v)$. Dijkstra's correctness comes entirely from *which order* we relax edges in — always from the closest unfinalized vertex first.

In [ ]:
# C06 — initialize_single_source and relax: the atomic primitives.
# Kept free of heap/drawing logic so C08 (heap) and C12 (array) can both reuse them.
def initialize_single_source(graph, s):
    if s not in graph.nodes():
        raise ValueError(f"source {s!r} not in graph")
    dist = {n: math.inf for n in graph.nodes()}
    pi = {n: None for n in graph.nodes()}
    dist[s] = 0
    return dist, pi

def relax(u, v, w, dist, pi):
    '''If u->v improves v's estimate, update dist/pi and return True; else False.

    math.inf + w stays inf, so an unreached u can never falsely relax v.
    No non-negative guard, so C14's Bellman-Ford oracle can reuse this directly.
    '''
    if u not in dist or v not in dist:
        raise KeyError(f"relax: {u!r} or {v!r} missing from dist")
    if dist[u] + w < dist[v]:
        dist[v] = dist[u] + w
        pi[v] = u
        return True
    return False

# Sanity check
_d, _p = initialize_single_source(example_graph, "s")
assert _d["s"] == 0 and _d["t"] == math.inf
assert relax("s", "y", 5, _d, _p) is True and _d["y"] == 5
assert relax("s", "y", 5, _d, _p) is False      # no improvement on repeat
assert _p["s"] is None and _p["y"] == "s"
print("Initialized dist for source 's':", {k: _fmt_dist(v) for k, v in _d.items()})

In [ ]:
# C07 — Interactive relaxation sanity check: isolate the relax decision.
edge_labels = [f"{u}->{v} (w={d['weight']})" for u, v, d in example_graph.edges(data=True)]

def _parse_edge(edge_label):
    head, rest = edge_label.split("->")
    u = head.strip()
    v = rest.split("(")[0].strip()
    w = int(rest.split("w=")[1].rstrip(") ").strip())
    return u, v, w

def relax_demo(edge_label, ud, vd):
    try:
        u, v, w = _parse_edge(edge_label)
    except Exception:
        print("Could not parse edge:", edge_label)
        return
    dist = {u: ud, v: vd}
    pi = {u: None, v: None}
    candidate = ud + w
    changed = relax(u, v, w, dist, pi)
    print(f"Edge {u} -> {v},   w({u},{v}) = {w}")
    print(f"  u.d = {ud},   v.d (before) = {vd}")
    print(f"  candidate = u.d + w = {ud} + {w} = {candidate}")
    print(f"  relax condition  (v.d > u.d + w)  ->  {vd} > {candidate}  ->  {vd > candidate}")
    print(f"  RELAXED: {changed}    v.d (after) = {dist[v]}")

if not edge_labels:
    print("No edges available to demo.")
else:
    interact(relax_demo,
             edge_label=Dropdown(options=edge_labels, description="edge"),
             ud=IntSlider(value=0, min=0, max=20, description="u.d"),
             vd=IntSlider(value=15, min=0, max=20, description="v.d"))

In [ ]:
# C08 — Dijkstra with a binary heap, recording a trace for animation.
def _snapshot(step, action, S, u, dist, pi, frontier):
    # Deep-ish copy of the mutable state so scrubbing never mutates other snapshots.
    return {
        "step": step,
        "action": action,
        "S": set(S),
        "u": u,
        "dist": dict(dist),
        "pi": dict(pi),
        "frontier": list(frontier),
    }

def dijkstra_with_trace(graph, s, weight="weight"):
    if s not in graph.nodes():
        raise ValueError(f"source {s!r} not in graph")
    dist, pi = initialize_single_source(graph, s)
    S = set()
    counter = itertools.count()          # tie-breaker so heap never compares nodes
    heap = [(0, next(counter), s)]
    trace = []
    step = 0
    trace.append(_snapshot(step, f"Initialize: source = {s}",
                           S, None, dist, pi, [n for _, _, n in heap]))
    while heap:
        du, _, u = heapq.heappop(heap)
        if u in S:
            continue                     # stale entry (lazy deletion)
        S.add(u)
        step += 1
        trace.append(_snapshot(step, f"Extract-Min: u = {u} (d = {_fmt_dist(du)})",
                               S, u, dist, pi, [n for _, _, n in heap if n not in S]))
        for _, v, data in graph.out_edges(u, data=True):
            if v in S:
                continue
            w = data.get(weight, 1)
            if relax(u, v, w, dist, pi):
                heapq.heappush(heap, (dist[v], next(counter), v))
                step += 1
                trace.append(_snapshot(step, f"Relax({u}, {v}): {v}.d = {_fmt_dist(dist[v])}",
                                       S, u, dist, pi, [n for _, _, n in heap if n not in S]))
    return dist, pi, trace

dist_ex, pi_ex, trace_steps = dijkstra_with_trace(example_graph, "s")
assert dist_ex == {"s": 0, "y": 5, "t": 8, "z": 7, "x": 9}, dist_ex
assert len(trace_steps) >= example_graph.number_of_nodes()
assert all(set(snap["S"]) <= set(example_graph.nodes()) for snap in trace_steps)
print("Final distances:", dist_ex)
print("Trace recorded:", len(trace_steps), "steps")

In [ ]:
# C09 — Animated step-by-step trace of Dijkstra on the example graph.
def render_step(i):
    if not trace_steps:
        print("No trace steps to display.")
        return
    i = max(0, min(i, len(trace_steps) - 1))   # clamp against a stale slider
    snap = trace_steps[i]
    S, u = snap["S"], snap["u"]
    Q = set(example_graph.nodes()) - S - ({u} if u is not None else set())
    plt.close("all")                            # fresh figure each frame (no ghosting)
    fig, ax = plt.subplots(figsize=(6, 5))
    try:
        draw_graph(example_graph, POS, dist=snap["dist"], pi=snap["pi"],
                   S=S, u=u, Q=Q,
                   title=f"Step {i}/{len(trace_steps)-1}: {snap['action']}", ax=ax)
    except Exception as e:
        ax.text(0.5, 0.5, f"draw error: {e}", ha="center")
    plt.show()
    print(f"S (finalized) = {sorted(S)}")
    print(f"u (current)   = {u}")
    print(f"Q (in queue)  = {sorted(Q)}")

step_slider = IntSlider(value=0, min=0, max=max(0, len(trace_steps) - 1), description="step")
play = Play(value=0, min=0, max=max(0, len(trace_steps) - 1), interval=900, description="play")
jslink((play, "value"), (step_slider, "value"))
try:
    display(widgets.HBox([play, step_slider]))
except NameError:
    pass
interact(render_step, i=step_slider)

In [ ]:
# C10 — Hands-on practice graph: predict the answers, then verify.
# Isomorphic to the example (t->a, y->b, x->c, z->d) plus a new vertex e.
PRACTICE_EDGES = [
    ("s", "a", 10), ("s", "b", 5),
    ("a", "c", 1),  ("a", "b", 2),
    ("b", "a", 3),  ("b", "c", 9), ("b", "d", 2),
    ("c", "d", 4),
    ("d", "c", 6),  ("d", "s", 7), ("d", "e", 3),
]
PRACTICE_POS = {
    "s": (0.0, 1.0),
    "a": (1.0, 2.0),
    "b": (1.0, 0.0),
    "c": (2.0, 2.0),
    "d": (2.0, 0.0),
    "e": (3.0, 1.0),
}
EXPECTED = {"s": 0, "a": 8, "b": 5, "c": 9, "d": 7, "e": 10}

def build_practice_graph():
    return build_graph(PRACTICE_EDGES)

practice_graph = build_practice_graph()
practice_dist, practice_pi, practice_steps = dijkstra_with_trace(practice_graph, "s")
assert set(practice_graph.nodes()) == {"s", "a", "b", "c", "d", "e"}
assert practice_dist == EXPECTED, f"{practice_dist} != {EXPECTED}"

def render_practice(i, reveal):
    if not practice_steps:
        print("No steps.")
        return
    i = max(0, min(i, len(practice_steps) - 1))
    snap = practice_steps[i]
    S, u = snap["S"], snap["u"]
    Q = set(practice_graph.nodes()) - S - ({u} if u is not None else set())
    plt.close("all")
    fig, ax = plt.subplots(figsize=(6, 5))
    draw_graph(practice_graph, PRACTICE_POS, dist=snap["dist"], pi=snap["pi"],
               S=S, u=u, Q=Q,
               title=f"Practice step {i}/{len(practice_steps)-1}: {snap['action']}", ax=ax)
    plt.show()
    if reveal:
        print(f"{'vertex':>6} | {'computed':>8} | {'expected':>8} | match")
        for n in sorted(EXPECTED):
            comp, exp = practice_dist[n], EXPECTED[n]
            print(f"{n:>6} | {comp:>8} | {exp:>8} |  {'OK' if comp == exp else 'X'}")
    else:
        print("Predict the final distance of each vertex, then tick 'reveal answer' to check.")

interact(render_practice,
         i=IntSlider(value=0, min=0, max=max(0, len(practice_steps) - 1), description="step"),
         reveal=Checkbox(value=False, description="reveal answer"))

## Running time depends on the priority queue

Dijkstra performs $|V|$ **Extract-Min** operations (one per vertex) and up to $|E|$ **Decrease-Key** operations (one per relaxation). The total cost is therefore:

$$T = |V| \cdot T_{\text{extract-min}} + |E| \cdot T_{\text{decrease-key}}$$

Plugging in the cost of each priority-queue implementation:

| Priority queue | Extract-Min | Decrease-Key | Total |
|---|---|---|---|
| **Linear array** | $O(V)$ | $O(1)$ | $O(V^2 + E) = O(V^2)$ |
| **Binary heap** | $O(\lg V)$ | $O(\lg V)$ | $O((V + E)\lg V) = O(E \lg V)$ |
| **Fibonacci heap** | $O(\lg V)$ amortized | $O(1)$ amortized | $O(E + V \lg V)$ |

- The **linear array** wins for **dense** graphs ($E \approx V^2$), where $O(V^2)$ beats $O(E \lg V) = O(V^2 \lg V)$.
- The **binary heap** wins for **sparse** graphs ($E = O(V)$), giving $O(V \lg V)$.
- The **Fibonacci heap** is asymptotically best because Decrease-Key is $O(1)$ amortized — but its large constants make it mostly of theoretical interest.

The next cell confirms the **array-vs-heap** trade-off empirically.

In [ ]:
# C12 — Array-based vs heap-based Dijkstra: empirical timing.
def dijkstra_array(graph, s, weight="weight"):
    '''Linear Extract-Min over the Q set: O(V^2).'''
    dist, pi = initialize_single_source(graph, s)
    Q = set(graph.nodes())
    while Q:
        u = min(Q, key=lambda n: dist[n])
        Q.remove(u)
        if dist[u] == math.inf:
            break                          # remaining vertices are unreachable
        for _, v, data in graph.out_edges(u, data=True):
            if v in Q:
                relax(u, v, data.get(weight, 1), dist, pi)
    return dist

def dijkstra_heap(graph, s, weight="weight"):
    '''Binary-heap variant (trace-free): O(E lg V).'''
    dist, pi = initialize_single_source(graph, s)
    S = set()
    counter = itertools.count()
    heap = [(0, next(counter), s)]
    while heap:
        _, _, u = heapq.heappop(heap)
        if u in S:
            continue
        S.add(u)
        for _, v, data in graph.out_edges(u, data=True):
            if v in S:
                continue
            if relax(u, v, data.get(weight, 1), dist, pi):
                heapq.heappush(heap, (dist[v], next(counter), v))
    return dist

# Both variants must agree with the answer key before we trust the timings.
assert dijkstra_array(example_graph, "s") == {"s": 0, "y": 5, "t": 8, "z": 7, "x": 9}
assert dijkstra_array(example_graph, "s") == dijkstra_heap(example_graph, "s")

def gen_random_graph(n, density, max_w, seed):
    if n < 2:
        raise ValueError("n must be >= 2")
    rng = random.Random(seed)
    g = nx.DiGraph()
    g.add_nodes_from(range(n))
    for u in range(n):
        for v in range(n):
            if u != v and rng.random() < density:
                g.add_edge(u, v, weight=rng.randint(1, max_w))
    for v in range(1, n):                   # backbone so node 0 reaches everything
        if not g.has_edge(v - 1, v):
            g.add_edge(v - 1, v, weight=rng.randint(1, max_w))
    return g

def time_algo(fn, graph, s, repeats=3):
    best = math.inf
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(graph, s)
        best = min(best, time.perf_counter() - t0)
    return best

def run_benchmark(sizes, density=0.2, max_w=20):
    results = {"array": [], "heap": []}
    print(f"{'n':>6} | {'array (ms)':>12} | {'heap (ms)':>12}")
    print("-" * 36)
    for n in sizes:
        g = gen_random_graph(n, density, max_w, seed=n)
        if dijkstra_array(g, 0) != dijkstra_heap(g, 0):
            print(f"WARNING: array/heap disagree at n={n}")
        ta = time_algo(dijkstra_array, g, 0)
        th = time_algo(dijkstra_heap, g, 0)
        results["array"].append(ta)
        results["heap"].append(th)
        print(f"{n:>6} | {ta*1000:>12.2f} | {th*1000:>12.2f}")
    return results

def run_experiment(max_n=400, density=0.2):
    sizes = [n for n in [50, 100, 200, 400, 800] if n <= max_n]
    results = run_benchmark(sizes, density, MAX_W)
    plt.figure(figsize=(6, 4))
    plt.plot(sizes, [t * 1000 for t in results["array"]], "o-", label="array  O(V^2)")
    plt.plot(sizes, [t * 1000 for t in results["heap"]], "s-", label="heap  O(E lg V)")
    plt.xlabel("number of vertices (n)")
    plt.ylabel("runtime (ms)")
    plt.title("Dijkstra: linear-array vs binary-heap priority queue")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    assert all(t >= 0 for t in results["array"] + results["heap"])
    return results

MAX_W = 20
# Default Colab-friendly run; drag the slider to push sizes higher.
interact(run_experiment,
         max_n=IntSlider(value=400, min=50, max=800, step=50, description="max n"),
         density=IntSlider(value=20, min=5, max=50, step=5, description="density %"))
print("Tip: the array curve (O(V^2)) climbs above the heap curve as n grows on sparse graphs.")

## Why it works: the loop invariant

> **Loop invariant.** At the start of each iteration of the main loop, for every vertex $v \in S$ we have $v.d = \delta(s, v)$ — every finalized estimate is already the true shortest distance.

**Proof sketch (induction on $|S|$).** Suppose $u$ is the vertex about to be extracted (the minimum-key vertex in $Q$), but for contradiction $u.d \ne \delta(s, u)$, i.e. $u.d > \delta(s, u)$. Consider a true shortest path $p$ from $s$ to $u$. Let $y$ be the **first** vertex on $p$ that lies *outside* $S$, and let $x \in S$ be its predecessor on $p$.

- Since $x \in S$, by the invariant $x.d = \delta(s, x)$, and the edge $x \to y$ was already relaxed when $x$ was finalized, so $y.d = \delta(s, y)$.
- $y$ appears **before** $u$ on a shortest path, so $\delta(s, y) \le \delta(s, u)$.
- **Here is where non-negativity matters:** the remaining sub-path from $y$ to $u$ has weight $\ge 0$, so $\delta(s, u) = \delta(s, y) + (\text{rest}) \ge \delta(s, y) = y.d$.

Chaining these: $y.d = \delta(s, y) \le \delta(s, u) \le u.d$. But Dijkstra chose $u$ as the minimum, so $u.d \le y.d$. Squeezing gives $u.d = \delta(s, u)$ — contradicting our assumption. Hence the invariant is maintained. $\blacksquare$

The one and only place the argument uses **"the rest of the path has weight $\ge 0$"** is the non-negativity assumption. Remove it and the proof collapses — which is exactly what the next cell demonstrates.

In [ ]:
# C14 — Negative-edge counterexample: why Dijkstra needs non-negative weights.
NEG_POS = {"s": (0.0, 1.0), "a": (1.6, 1.7), "b": (1.6, 0.3)}

def build_neg_graph(neg_weight):
    # True dist[a] = min(1, 2 + neg_weight). With neg_weight < -1 the cheaper
    # route s->b->a uses the negative edge, but Dijkstra finalizes a too early.
    g = nx.DiGraph()
    g.add_edge("s", "a", weight=1)
    g.add_edge("s", "b", weight=2)
    g.add_edge("b", "a", weight=neg_weight)
    return g

def bellman_ford(graph, s, weight="weight"):
    '''Correctness oracle: V-1 relaxation passes + a negative-cycle check.'''
    dist, pi = initialize_single_source(graph, s)
    for _ in range(graph.number_of_nodes() - 1):
        for u, v, data in graph.edges(data=True):
            relax(u, v, data.get(weight, 1), dist, pi)
    for u, v, data in graph.edges(data=True):
        if dist[u] + data.get(weight, 1) < dist[v]:
            raise ValueError("negative-weight cycle detected")
    return dist

def compare_neg(neg_weight):
    g = build_neg_graph(neg_weight)
    dij, _, _ = dijkstra_with_trace(g, "s")
    true_dist = bellman_ford(g, "s")
    plt.close("all")
    fig, ax = plt.subplots(figsize=(5, 4))
    draw_graph(g, NEG_POS, dist=true_dist, S=set(g.nodes()),
               title=f"b -> a weight = {neg_weight}", ax=ax)
    plt.show()
    print(f"{'vertex':>7} | {'Dijkstra':>9} | {'true (BF)':>9} | status")
    print("-" * 42)
    any_wrong = False
    for n in sorted(g.nodes()):
        d_val, t_val = dij[n], true_dist[n]
        wrong = d_val != t_val
        any_wrong = any_wrong or wrong
        print(f"{n:>7} | {_fmt_dist(d_val):>9} | {_fmt_dist(t_val):>9} | {'WRONG' if wrong else 'ok'}")
    if any_wrong:
        print("\nDijkstra finalized 'a' at d=1 via s->a before discovering the cheaper")
        print("s->b->a path (which uses the negative edge). Once a vertex enters S it is")
        print("never revisited, so the negative edge can never repair the estimate.")
    else:
        print("\nAll weights non-negative here: Dijkstra and Bellman-Ford agree.")

# The canonical counterexample: s->a=1, s->b=2, b->a=-2  =>  true dist[a]=0, Dijkstra says 1.
compare_neg(-2)
_dij, _, _ = dijkstra_with_trace(build_neg_graph(-2), "s")
_true = bellman_ford(build_neg_graph(-2), "s")
assert _dij != _true
assert _true["a"] == 0 and _dij["a"] == 1

interact(compare_neg, neg_weight=IntSlider(value=-2, min=-2, max=5, description="b->a w"))

## Summary and next steps

You built Dijkstra's algorithm end-to-end and saw each objective in action:

- **Relaxation (C05–C07).** The atomic update $v.d = \min(v.d,\ u.d + w(u,v))$ is the only operation that changes an estimate; the interactive demo isolated exactly when it fires.
- **The greedy main loop (C08–C10).** `dijkstra_with_trace` repeatedly extracts the closest unfinalized vertex, finalizes it into $S$, and relaxes its edges — and the scrubbable animation let you watch $S$, $u$, and $Q$ evolve on both the example and practice graphs.
- **Complexity (C11–C12).** The priority queue sets the running time: $O(V^2)$ array, $O(E\lg V)$ binary heap, $O(E + V\lg V)$ Fibonacci heap. The benchmark confirmed the array-vs-heap trade-off empirically.
- **Correctness (C13–C14).** The loop invariant $\big(\forall v \in S:\ v.d = \delta(s,v)\big)$ holds **only** because edge weights are non-negative — and the negative-edge counterexample showed Dijkstra returning a provably wrong answer the moment that assumption breaks.

### The non-negative-weight requirement

This is the headline caveat: **Dijkstra is correct iff every edge weight is $\ge 0$.** Early finalization is what makes it fast and what makes it fail on negative edges.

### Where to go next

- **Bellman-Ford** — handles negative edges (and detects negative cycles) in $O(VE)$; you already used it as the oracle in C14.
- **A\*** — Dijkstra plus an admissible heuristic to guide the search toward a target.
- **Prim's MST** — the same priority-queue skeleton, minimizing a different key (single edge weight rather than path distance).